In [56]:
from fredapi import Fred
from dotenv import load_dotenv
import os
import pandas as pd

project_root = '/Users/namansoni/Project/macro-regime-classifier'

load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
fred = Fred(api_key = FRED_API_KEY)

#Creating the dictionary holding the key series that define economic health
series_ids = {
    'cpi': 'CPIAUCSL',
    'yield_curve': 'T10Y2Y',
    'unemployment': 'UNRATE',
    'credit_spread': 'BAA10Y',
    'gdp_growth': 'A191RL1Q225SBEA',
    'consumer_sent': 'UMCSENT',
    'indust_prod': 'INDPRO',
}
print(series_ids)

{'cpi': 'CPIAUCSL', 'yield_curve': 'T10Y2Y', 'unemployment': 'UNRATE', 'credit_spread': 'BAA10Y', 'gdp_growth': 'A191RL1Q225SBEA', 'consumer_sent': 'UMCSENT', 'indust_prod': 'INDPRO'}


In [57]:
#Pulling the monthly data from FRED for each of the series in the series_ids dictionary starting from January 2001 to the most recent data
raw = {}
for name, series_id in series_ids.items():
    raw[name] = fred.get_series(series_id, observation_start = '2000-01-01')
    print(f"{name}: {len(raw[name])} observations | start: {raw[name].index[0].date()} | end: {raw[name].index[-1].date()}")

cpi: 317 observations | start: 2000-01-01 | end: 2026-05-01
yield_curve: 6918 observations | start: 2000-01-03 | end: 2026-07-08
unemployment: 318 observations | start: 2000-01-01 | end: 2026-06-01
credit_spread: 6917 observations | start: 2000-01-03 | end: 2026-07-07
gdp_growth: 105 observations | start: 2000-01-01 | end: 2026-01-01
consumer_sent: 317 observations | start: 2000-01-01 | end: 2026-05-01
indust_prod: 317 observations | start: 2000-01-01 | end: 2026-05-01


In [60]:
#Since there are different reportings of each series, the data must be manipulated such that it can be analyzed properly. We will be using a new dictionary to ensure the original data isn't damaged/touched
features = {}
# 1. CPI alone doesn't tell us much, so we need to calculate how much it has increased since the same time the year before to understand how fast prices are rising: 
features['cpi'] = raw['cpi'].pct_change(12)
# 2. Employment must be converted similarly to give a month to month change. However this time we use the .diff(n) with n = 1 method to find the difference between each row.
features['unemployment'] = raw['unemployment'].diff(1)
# 3. The GDP is only reported quarterly. But to match the rest of the data (which is monthly) to be able to use the data adequately. So we'll linearly interpolate from quarterly to monthly to fill in the missing values
# First we add NaN values for all the months between the gdp reportings using the resample method on ('ME') Month End frequency, so add a value for each month. Then the interpolate method is used to linearly fill those NaN values
features['gdp_growth'] = raw['gdp_growth'].resample('ME').first().interpolate(method = 'linear')
# 4/5. Yield Curve data and Credit spread must be resampled from having data for every day to having data for every month, so it can be used with the other series that have been resampled to monthly. 
features['yield_curve'] = raw['yield_curve'].resample('ME').last()
features['credit_spread'] = raw['credit_spread'].resample('ME').last()
# 6. Similar to Unemployment, the consumer sentiment and industrial production must be added to features as differences from the previous row.
features['consumer_sent'] = raw['consumer_sent'].diff(1)
features['indust_prod'] = raw['indust_prod'].diff(1)
#Writing a sanity check to make sure its working properly
for name, series in features.items():
    print(f"{name}: {len(features[name])} observations | start: {features[name].index[0].date()} | end: {features[name].index[-1].date()} | NaN count: {features[name].isna().sum()}")
    print(series.dropna().head(3))

cpi: 317 observations | start: 2000-01-01 | end: 2026-05-01 | NaN count: 13
2001-01-01    0.037212
2001-02-01    0.035294
2001-03-01    0.029825
dtype: float64
unemployment: 318 observations | start: 2000-01-01 | end: 2026-06-01 | NaN count: 3
2000-02-01    0.1
2000-03-01   -0.1
2000-04-01   -0.2
dtype: float64
gdp_growth: 313 observations | start: 2000-01-31 | end: 2026-01-31 | NaN count: 0
2000-01-31    1.5
2000-02-29    3.5
2000-03-31    5.5
Freq: ME, dtype: float64
yield_curve: 319 observations | start: 2000-01-31 | end: 2026-07-31 | NaN count: 0
2000-01-31    0.07
2000-02-29   -0.11
2000-03-31   -0.47
Freq: ME, dtype: float64
credit_spread: 319 observations | start: 2000-01-31 | end: 2026-07-31 | NaN count: 0
2000-01-31    1.68
2000-02-29    1.91
2000-03-31    2.30
Freq: ME, dtype: float64
consumer_sent: 317 observations | start: 2000-01-01 | end: 2026-05-01 | NaN count: 1
2000-02-01   -0.7
2000-03-01   -4.2
2000-04-01    2.1
dtype: float64
indust_prod: 317 observations | start: 2

In [61]:

# Now this raw data must be saved into the data folder for access in the future. This is so that we don't have to do a api call each time we run the file or run any method. A CSV file will be created by first converting the dictionary (features) into a dataframe.
raw_df = pd.DataFrame(features)
print(raw_df.head(3))
raw_df.to_csv(os.path.join(project_root, 'data', 'macro_features_raw.csv'))

# Important note: This csv is my features before alignment (which will be done tomorrow) 

            cpi  unemployment  gdp_growth  yield_curve  credit_spread  \
2000-01-01  NaN           NaN         NaN          NaN            NaN   
2000-01-31  NaN           NaN         1.5         0.07           1.68   
2000-02-01  NaN           0.1         NaN          NaN            NaN   

            consumer_sent  indust_prod  
2000-01-01            NaN          NaN  
2000-01-31            NaN          NaN  
2000-02-01           -0.7       0.2859  
